# Homework 2 - PCA

TODO Assignment Summary

In [200]:
''' Dependencies required for this assignment '''
from pprint import pprint  # Pretty printing
import pandas as pd  # Dataframe management
import numpy as np  # Numrical computations
import pandas as pd  # Data manipulation and analysis
import numpy as np  # Numerical Computing
import matplotlib.pyplot as plt  # Data Visualization with Static Plots
import plotly.graph_objects as go  # Interative Data Visualization
from sklearn.decomposition import PCA  # Dimensionality Reduction
from sklearn.preprocessing import StandardScaler, MinMaxScaler  # Data standardization and normalization
from sklearn.model_selection import train_test_split  # Data Splitting
from sklearn.neighbors import KNeighborsClassifier  # ML Models
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report  # Evaluating Model Performance

## Problem 1 - Loading the Dataset
Loading the breast cancer dataset from the provided *csv* file.

In [201]:
df_bc = pd.read_csv('./data/breast-cancer-wisconsin.data')
df_bc

,1000025,5,1,1.1,1.2,2,1.3,3,1.4,1.5,2.1
0,1002945,5,4,4,5,7,10,3,2,1,2
1,1015425,3,1,1,1,2,2,3,1,1,2
2,1016277,6,8,8,1,3,4,3,7,1,2
3,1017023,4,1,1,3,2,1,3,1,1,2
4,1017122,8,10,10,8,7,10,9,7,1,4
...,...,...,...,...,...,...,...,...,...,...,...
693,776715,3,1,1,1,3,2,1,1,1,2
694,841769,2,1,1,1,2,1,1,1,1,2
695,888820,5,10,10,3,7,3,8,10,2,4
696,897471,4,8,6,4,3,4,10,6,1,4


## Problem 2 - Cleaning the Dataset

Creating a new dataframe containing only records with complete data, verifying that the new (cleaned) dataframe contains only records with complete data records.

When initially cleaning the dataset, we should note that (as stated in the .names file of the data) some incomplete samples are filled with a "?". This will not be covered by panda's `dropna()` function, so we should firstly replace then values with NaN values.

In [202]:
df_bc = df_bc.replace('?', np.nan)  # Replace all '?' values with numpy's NaN to drop

Now, we can take the approach of dropping incomplete samples and perform dropna() as normal.

In [203]:
clean_df = df_bc.dropna(how="any")  # Drop any NA values
print(f"Incomplete Records:\n{clean_df.isna().sum()}")  # Verify all records are complete
clean_df = clean_df.drop_duplicates()  # Drop any duplicate values
clean_df.info()  # Printing essential info for df evalulation

Incomplete Records:
1000025    0
5          0
1          0
1.1        0
1.2        0
2          0
1.3        0
3          0
1.4        0
1.5        0
2.1        0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 674 entries, 0 to 697
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   1000025  674 non-null    int64 
 1   5        674 non-null    int64 
 2   1        674 non-null    int64 
 3   1.1      674 non-null    int64 
 4   1.2      674 non-null    int64 
 5   2        674 non-null    int64 
 6   1.3      674 non-null    object
 7   3        674 non-null    int64 
 8   1.4      674 non-null    int64 
 9   1.5      674 non-null    int64 
 10  2.1      674 non-null    int64 
dtypes: int64(10), object(1)
memory usage: 63.2+ KB


## Problem 3 - Pre-processing

Creating a new column "target" containing integer re-categorizations of the original Class field.

When first attempting this problem, it seemed intuitive to map each unique target to a number, such as:

```py
unique_raw_target_vals = clean_df.iloc[:,-1].unique()
for i in range(len(unique_raw_target_vals)):
   clean_df.loc[clean_df.iloc[:,-1] == unique_raw_target_vals[i], 'target'] = i

```

However, this would not work as the target is already in numerical values. Therefore, one approach could be to **normalize** the data, essentially mapping each value to 0 or 1. 

In [204]:
normalize_scaler = MinMaxScaler()  # Create a min-max scaler (normalization)
# Create a new column of the normalized target values as ints
clean_df['target'] = normalize_scaler.fit_transform(clean_df.iloc[:,[-1]]).astype("int32")
clean_df['target'].unique()

array([0, 1])

Alternatively, we achieve the same result by using panda's `factorize` functionality [[1]](https://pandas.pydata.org/docs/reference/api/pandas.factorize.html#pandas.factorize), which maps categorical data into integer representations (such as 0, 1, 2, 3, ...). In this case, it will simply map 2 -> 0 and 4 -> 1.

In [205]:
clean_df_copy = clean_df.copy()
clean_df_copy['target'] = pd.factorize(clean_df_copy.iloc[:, -1])[0].astype("int32")
clean_df_copy['target'].unique()

array([0, 1])

Additionally, we must consider the feature data. In order to account for different scales and preserve outliers, we can **standardize** the data (excluding the first column of IDs, which is purely an identifier and will *not* be considered as a feature).

In [206]:
# Casting feature columns to floats to avoid future depracation warnings
cols = clean_df.columns[1:-2]  # Extract the specific columns to easier for-loop
clean_df = clean_df.astype({c: 'float64' for c in cols})  # Convert each feature datatype to a float

# Saving an un-standardized copy for later steps
unstandardized_clean_df = clean_df.copy()

# Applying standardization to the features
standardize_scaler = StandardScaler()  # Making a standardization scaler
scaled_features: np.array = standardize_scaler.fit_transform(clean_df[cols])  # Applying the standardization transformation
clean_df[cols] = pd.DataFrame(scaled_features, columns=cols)  # Mapping to numpy array back to the dataframe
clean_df

,1000025,5,1,1.1,1.2,2,1.3,3,1.4,1.5,2.1,target
0,1002945,0.194613,0.278383,0.264788,0.747250,1.706778,1.775982,-0.180787,-0.289983,-0.349432,2,0
1,1015425,-0.514410,-0.703973,-0.743502,-0.644038,-0.557727,-0.423902,-0.180787,-0.616274,-0.349432,2,0
2,1016277,0.549125,1.588190,1.609175,-0.644038,-0.104826,0.126069,-0.180787,1.341474,-0.349432,2,0
3,1017023,-0.159899,-0.703973,-0.743502,0.051606,-0.557727,-0.698887,-0.180787,-0.616274,-0.349432,2,0
4,1017122,1.258149,2.243094,2.281369,1.790715,1.706778,1.775982,2.264366,1.341474,-0.349432,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...
693,776715,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0
694,841769,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0
695,888820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,1
696,897471,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,1


## Problem 4 - Feature/Target Data Split

Next, we can create X and Y splits similarly to how we did before, excluding the ID field.

In [207]:
feature_cols = clean_df.columns[1:-2]  # Explicitly state the feature columns
target_col = clean_df.columns[-1]  # Explicitly state the target column
# Splitting the datasets
X = clean_df[feature_cols]
y = clean_df[target_col]
# Verifications
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"X column names: {X.columns}")

X shape: (674, 9), y shape: (674,)
X column names: Index(['5', '1', '1.1', '1.2', '2', '1.3', '3', '1.4', '1.5'], dtype='object')


## Problem 5 - Standardization of Feature Data

Although this was already achieved in the pre-processing step, we can break the logic out into a function and verify the results. This way, we ensure that *only the training data* gets standardized.

In [ ]:
# Defining a function to standardize all given columns from a dataframe
def standardize_df(
                    df: pd.DataFrame, 
                    cols: pd.Index = None, 
                    inPlace: bool = False
                  ) -> pd.DataFrame | None :
    ''' 
    Standardizes a given dataframe at their given column indexes
        - df: input dataframe
        - cols: the column names to scale. If none, assumes all columns
        - inPlace: If true, modify the df and return None; else, return a new dataframe
    '''
    # Work on a copy unless inPlace is true
    out_df = df if inPlace else df.copy()
    
    # Set cols to equal all columns if no argument is received
    if cols is None:
        cols = df.columns
    
    # Convert each feature datatype to a float
    out_df.loc[:, cols] = (out_df.loc[:, cols].astype("float64"))  # Workaround to avoid copying
    
    # Fit/transform with the standard scaler 
    standardize_scaler = StandardScaler()  # Define the standardization scaler
    scaled_features = standardize_scaler.fit_transform(out_df[cols])  # Fit to the features
    
    # Assign the scaled features back to a dataframe
    out_df[cols] = pd.DataFrame(scaled_features, columns=cols)
    
    return None if inPlace else out_df

# Example usage of function using copy of unstandaridized data from earlier
X = unstandardized_clean_df.loc[:, feature_cols].copy()  # Deep copy
standardize_df(X, cols=feature_cols, inPlace=True)

# Verifications - should be the same, where mean=0 and std=1
print("Description (if standardized, mean ~= 0 and standard deviation ~= 1")
X.describe()

Description (if standardized, mean ~= 0 and standard deviation ~= 1


,5,1,1.1,1.2,2,1.3,3,1.4,1.5
count,651.000000,651.000000,651.000000,651.000000,651.000000,651.000000,651.000000,651.000000,651.000000
mean,-0.006331,-0.012853,-0.011420,-0.000220,-0.012298,-0.014592,-0.002377,-0.011808,-0.021185
std,0.996506,0.989553,0.993998,1.004806,0.983604,0.995027,1.004129,0.992652,0.978334
min,-1.223434,-0.703973,-0.743502,-0.644038,-1.010628,-0.698887,-0.995838,-0.616274,-0.349432
25%,-0.868922,-0.703973,-0.743502,-0.644038,-0.557727,-0.698887,-0.588312,-0.616274,-0.349432
50%,-0.159899,-0.703973,-0.743502,-0.644038,-0.557727,-0.698887,-0.180787,-0.616274,-0.349432
75%,0.549125,0.442109,0.600885,0.399428,0.348075,0.538547,0.634264,0.362600,-0.349432
max,1.967172,2.243094,2.281369,2.486359,3.065481,1.775982,2.671892,2.320348,4.820460


## Problem 6 - Training a $k$-NN Model

First, we will need to split the data into training and testing sets.

In [215]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

Then, we can define a function to test the $k$-NN model from $k ∈ {1, ..., 100}$.

In [219]:
def get_kNN_accuracy(
                      X_train: pd.DataFrame, 
                      X_test: pd.DataFrame, 
                      y_train: pd.DataFrame, 
                      y_test: pd.DataFrame,
                      k_val: int 
                    ) -> float:
    ''' Returns the accuracy of a given k for a k-NN model, given the train/test splits '''
    y_pred = KNeighborsClassifier(n_neighbors=k_val).fit(X_train, y_train).predict(X_test)  # Train the model with k nearest neighbors and get the test prediction
    return accuracy_score(y_test, y_pred)  # Return the accuracy score of that prediction

Now, we can use this function to test $k$, storing the results to visualize.

In [220]:
k_results = dict()  # Dictionary data structure (hashmap) to store results
k_optimal = 1  # Optimal k value
k_best_acc = 0  # Optimal k accuracy

# Retreive and store predictions for k ∈ {1, ..., 100
for k in range(1, 101):
    cur_acc = get_kNN_accuracy(X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, k_val=k)
    k_results[k] = cur_acc
    if k_best_acc < cur_acc:
        k_best_acc = cur_acc
        k_optimal = k

ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values